In [3]:
import json
import os
from pathlib import Path

from helpers import load_env, mint_token, vb, append_to_env, voice_widget

load_env()
assert os.getenv("VOCAL_BRIDGE_API_KEY"), "Set VOCAL_BRIDGE_API_KEY in .env"
print("VB API key loaded ✓")

VB API key loaded ✓


In [4]:
print(Path("agents/voice-app/prompt.md").read_text())

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 436: character maps to <undefined>

In [5]:
actions = json.loads(Path("agents/voice-app/actions.json").read_text())
for a in actions:
    arrow = "→" if a["direction"] == "agent_to_app" else "←"
    print(f"  {arrow} {a['name']:<22} {a['description']}")

  → show_tic_tac_toe       Render the board. Fire to start a new game. Payload: {userSymbol: 'X'|'O', firstTurn: 'user'|'agent'}.
  → place_mark             Place YOUR (agent's) mark at row/col. Fire on every agent turn. Payload: {row: 0-2, col: 0-2, speech?: string}.
  → user_move              Place the LEARNER's mark when they SPOKE the move. Fire in same response as place_mark. Payload: {row, col, heard}.
  → clear_ui               Reset the board to idle.
  ← user_placed_mark       Learner CLICKED a cell. Payload has the new full board, status, turn, plus the move. Respond with your place_mark and one short reaction.
  ← board_sync             Silent authoritative board state after every placement. Payload: {board, status, turn, moveCount, userSymbol, agentSymbol}. Use as ground truth.


In [6]:
import os

# Idempotent: if an agent ID is already set, reuse it. Otherwise create one.
agent_id = os.environ.get("VOCAL_BRIDGE_AGENT_ID", "").strip()
if agent_id:
    print(f"Using pre-provisioned agent: {agent_id}")
else:
    result = vb(
        "agent", "create",
        "--name", "Tic_Tac_Toe-voice-app",
        "--style", "Chatty",
        "--prompt-file", "agents/voice-app/prompt.md",
        "--client-actions-file", "agents/voice-app/actions.json",
        "--deploy-targets", "web",
        json_output=True,
    )
    agent = result["agent"]
    agent_id = agent["id"]
    append_to_env("VOCAL_BRIDGE_AGENT_ID", agent_id)
    print(f"Created '{agent['name']}'  ({agent['mode']})")
    print(f"  id: {agent_id}")
    print(f"  saved to .env as VOCAL_BRIDGE_AGENT_ID")

Using pre-provisioned agent: 1e61deef-fdc2-4e15-8a9b-76443146b63e


In [7]:
token = mint_token(agent_id, participant_name="Notebook Learner")
print(f"session_name:   {token.session_name}")
print(f"agent_mode:     {token.agent_mode}")
print(f"token (truncd): {token.token[:40]}…")
# token.connection_url is the wss:// endpoint the SDK opens — kept off-screen to keep this clean.


session_name:   user-a6d33e19-agent-1e61deef-api-967f5170
agent_mode:     openai_concierge
token (truncd): eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ…


In [8]:
from IPython.display import HTML
HTML(voice_widget(token))

In [7]:
vb("agent", "use", agent_id)
logs = vb("logs", "--limit", "1", json_output=True)
if not logs:
    print("No calls yet — try the widget above first.")
else:
    print(vb("logs", "show", logs['sessions'][0]['id']))

Session: 9464e179-c06a-410a-b069-b79dd6f7ca8f
Status: completed
Started: 2026-07-14 20:55:03
Ended: 2026-07-14 20:56:40
Duration: 1m 36s
Messages: 9
Direction: web
Recording: Available (use 'vb logs download 9464e179-c06a-410a-b069-b79dd6f7ca8f' to download)

--- Transcript ---
AGENT: Hey — I'm a Voice agent. Say 'let's play tic-tac-toe' to start.

USER: Let's play tic-tac-toe

TOOL CALL: trigger_client_action

TOOL OUTPUT (trigger_client_action): Action 'show_tic_tac_toe' triggered successfully. The client application will handle this action.

AGENT: You're X — your move.

USER: [Client Event] The user triggered 'user_placed_mark' with data: {"board": ["", "", "", "", "X", "", "", "", ""], "status": "playing", "turn": "agent", "moveCount": 1, "userSymbol": "X", "agentSymbol": "O", "position": 4, "row": 1, "col": 1, "mark": "X"}

TOOL CALL: trigger_client_action

TOOL OUTPUT (trigger_client_action): Action 'place_mark' triggered successfully. The client application will handle this act